# Hindi Audio Transcription — Whisper large-v3 (Google Colab)

Transcribes **Hindi spiritual / devotional audio** (up to ~1 hour each) using OpenAI's `whisper-large-v3` via **faster-whisper** (fast, GPU-friendly).

**How to use:**
1. Open this notebook in Google Colab.
2. Set the runtime to GPU: `Runtime → Change runtime type → T4 GPU` (or better).
3. Run each cell top to bottom. Cell **4** lets you pick files from your local drive.
4. Outputs (`.txt`, `.srt`, `.vtt`) are saved and zipped for download at the end.


## 1. Check the GPU
If this shows a GPU (e.g. Tesla T4), you're good. If it errors, switch the runtime to GPU first.

In [ ]:
!nvidia-smi

## 2. Install dependencies
`faster-whisper` runs the large-v3 model several times faster than the reference implementation and comfortably handles 1-hour files on a free T4.

In [ ]:
!pip install -q faster-whisper==1.0.3
print('done')

## 3. Settings
Tweak these if needed. The `INITIAL_PROMPT` biases the model toward devotional/spiritual vocabulary and correct spelling of common terms — edit it to match your content (add guru names, mantras, place names, etc.).

In [ ]:
LANGUAGE = "hi"          # Hindi
MODEL_SIZE = "large-v3"  # Whisper large-v3

# Devotional/spiritual context hint. Helps with proper nouns & terminology.
# Keep it in Hindi (Devanagari) so the model stays in Hindi script.
INITIAL_PROMPT = (
    "यह एक आध्यात्मिक और भक्ति प्रवचन है। इसमें ईश्वर, गुरु, भक्ति, ध्यान, मंत्र, भजन, सत्संग "
    "और शास्त्रों की चर्चा होती है।"
)

# Quality / speed knobs
BEAM_SIZE = 5            # higher = a bit more accurate, a bit slower
VAD_FILTER = True        # skip long silences (good for chant/discourse gaps)
WORD_TIMESTAMPS = False  # set True if you need word-level timing in SRT

print(f"Model={MODEL_SIZE}  lang={LANGUAGE}  beam={BEAM_SIZE}")

## 4. Pick files from your local drive
Run this cell, then choose one or more audio files from your computer (mp3, wav, m4a, flac, etc.). They upload into the Colab session.

In [ ]:
import os
from google.colab import files

os.makedirs("audio_in", exist_ok=True)
os.makedirs("transcripts_out", exist_ok=True)

uploaded = files.upload()  # opens a file picker for your local drive

for name, data in uploaded.items():
    dest = os.path.join("audio_in", name)
    with open(dest, "wb") as f:
        f.write(data)
    print(f"saved -> {dest}  ({len(data)/1e6:.1f} MB)")

audio_files = sorted(
    os.path.join("audio_in", n) for n in os.listdir("audio_in")
)
print(f"\n{len(audio_files)} file(s) ready.")

## 5. Load the model
Downloads `large-v3` the first time (~3 GB). Uses `float16` on GPU for speed.

In [ ]:
import torch
from faster_whisper import WhisperModel

if torch.cuda.is_available():
    device, compute_type = "cuda", "float16"
else:
    device, compute_type = "cpu", "int8"
    print("WARNING: no GPU detected — this will be very slow. Switch runtime to GPU.")

print(f"Loading {MODEL_SIZE} on {device} ({compute_type})...")
model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)
print("Model ready.")

## 6. Transcribe
Writes a `.txt` (plain transcript), `.srt` and `.vtt` (subtitles with timestamps) per file into `transcripts_out/`.

In [ ]:
import time
from pathlib import Path

def fmt_ts(seconds, sep=","):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"

for path in audio_files:
    stem = Path(path).stem
    print(f"\n=== {stem} ===")
    t0 = time.time()

    segments, info = model.transcribe(
        path,
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        vad_filter=VAD_FILTER,
        word_timestamps=WORD_TIMESTAMPS,
        initial_prompt=INITIAL_PROMPT,
    )
    print(f"duration={info.duration:.0f}s  detected_lang={info.language} ({info.language_probability:.2f})")

    txt_lines, srt_lines, vtt_lines = [], [], ["WEBVTT", ""]
    for i, seg in enumerate(segments, 1):
        text = seg.text.strip()
        txt_lines.append(text)
        srt_lines.append(
            f"{i}\n{fmt_ts(seg.start)} --> {fmt_ts(seg.end)}\n{text}\n"
        )
        vtt_lines.append(
            f"{fmt_ts(seg.start, '.')} --> {fmt_ts(seg.end, '.')}\n{text}\n"
        )
        # live progress (handy for long files)
        if i % 25 == 0:
            print(f"  ...{i} segments, up to {seg.end:.0f}s")

    Path(f"transcripts_out/{stem}.txt").write_text("\n".join(txt_lines), encoding="utf-8")
    Path(f"transcripts_out/{stem}.srt").write_text("\n".join(srt_lines), encoding="utf-8")
    Path(f"transcripts_out/{stem}.vtt").write_text("\n".join(vtt_lines), encoding="utf-8")
    print(f"  done in {time.time()-t0:.0f}s -> transcripts_out/{stem}.txt/.srt/.vtt")

print("\nAll files transcribed.")

## 7. Preview a transcript

In [ ]:
from pathlib import Path
txts = sorted(Path("transcripts_out").glob("*.txt"))
if txts:
    print(f"--- {txts[0].name} (first 1500 chars) ---\n")
    print(txts[0].read_text(encoding="utf-8")[:1500])

## 8. Download results
Zips everything and downloads it back to your local drive.

In [ ]:
from google.colab import files
!zip -r -q transcripts.zip transcripts_out
files.download("transcripts.zip")